# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoders:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)
- Decoders:
    - Qwen3-0.6B (Qwen/Qwen3-0.6B) (0.6B parameters)
    - Llama-3.2-1B (meta-llama/Llama-3.2-1B) (1B parameters)

In [1]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets.dataset_dict import DatasetDict
from datasets.arrow_dataset import Dataset
from datasets import load_dataset

import datasets
import os

## Baseline models evaluation

In [2]:
from transformers import AutoTokenizer, AutoModel

In [7]:
xml_roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

def chunk_text(text, chunk_size=512, overlap=62):
    token_ids = xml_roberta_tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    chunks = []
    start = 0
    while start < len(token_ids):
        end = start + chunk_size
        chunk_tokens = token_ids[start:end]
        chunk_text = xml_roberta_tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start += chunk_size - overlap

    return chunks



def get_embeddings_for_chunked_texts(
        inputs,    # a sequence of texts
        tokenizer,
        model,
        max_length):
    embeddings = []

    # for each text in input make a chunking
    texts_chunked = [chunk_text(text) for text in inputs]
    texts_chunked_tokenized = []

    for text in texts_chunked:    # for each text in chunked input texts
        # perform a tokeniztaion of each chunk
        chunk_input_ids = []
        chunk_attention_masks = []

        tokenized = tokenizer(
            text,    # text is effectively a batch of chunks at this point
            padding="longest",
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        # get embeddings for all chunks
        with torch.no_grad():
            outputs = model(**tokenized)
            # mean pooling for each chunk
            embedding = outputs.last_hidden_state.mean(dim=1)
            # mean pooling across chunks:
            embedding = embedding.mean(dim=0)
            embeddings.append(embedding.unsqueeze(0).detach())

    return_embed = torch.cat(embeddings, dim=0)
    return return_embed

In [79]:
class XMLRoBERTa:

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")
        self.tokenizer = AutoTokenizer.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            device_map="auto")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        return_embed = get_embeddings_for_chunked_texts(
            inputs,
            self.tokenizer,
            self.model,
            self.chunk_size
        )
        return return_embed

In [80]:
class Qwen3_Embedding:

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            device_map="auto")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        return_embed = get_embeddings_for_chunked_texts(
            inputs,
            self.tokenizer,
            self.model,
            self.chunk_size
        )
        return return_embed

In [ ]:
class Qwen3:

    name = "Qwen/Qwen3-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("Qwen//Qwen3-0.6B")
        self.tokenizer = AutoTokenizer.from_pretrained(
            "Qwen//Qwen3-0.6B",
            dtype=torch.float16,
            device_map="auto")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        return_embed = get_embeddings_for_chunked_texts(
            inputs,
            self.tokenizer,
            self.model,
            self.chunk_size
        )
        return return_embed

In [ ]:
class Llama3_2:

    name = "meta-llama/Llama-3.2-1B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B")
        self.tokenizer = AutoTokenizer.from_pretrained(
            "meta-llama/Llama-3.2-1B",
            dtype=torch.float16,
            device_map="auto")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:

        return_embed = get_embeddings_for_chunked_texts(
            inputs,
            self.tokenizer,
            self.model,
            self.chunk_size
        )
        return return_embed

### Testing on example data

In [3]:
model = AutoModel.from_pretrained("Qwen/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-0.6B",
    dtype=torch.float16,
    device_map="auto")

In [9]:
chunked = chunk_text([test_text])

for text in chunked:    # for each text in chunked input texts
    # perform a tokeniztaion of each chunk
    chunk_input_ids = []
    chunk_attention_masks = []

    tokenized = tokenizer(
        text,    # text is effectively a batch of chunks at this point
        padding="longest",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    # get embeddings for all chunks
    with torch.no_grad():
        outputs = model(**tokenized)
        # mean pooling for each chunk
        embedding = outputs.last_hidden_state.mean(dim=1)
        # mean pooling across chunks:
        embedding = embedding.mean(dim=0)

In [11]:
embedding.size()

torch.Size([1024])

In [4]:
data = load_dataset("dwzhu/LongEmbed", 'qmsum')

In [5]:
test_text = data["corpus"]["text"][0]
test_text[:1000]

"Project Manager: Can I close this ?\nUser Interface: Uh we don't have any changes , do we ?\nProject Manager: Oh , okay .\nUser Interface: So no . {vocalsound}\nProject Manager: {vocalsound} There we go . Okay , here we are again . Detailed design {disfmarker} oh , come on . Well {disfmarker} Ah {gap} s Forgot to insert the minutes , but it's about the same thing we discussed before . Uh {disfmarker} Could open that anyway , think . Other design {disfmarker} anyway , we took as {disfmarker} we took w we took rubber as as the material last time . We also {gap} that you're just busy with it . Took the advanced chip to t uh implement the advanced features . Well , we discussed the design , no sharp corners , we rounded it off , like you see on the {gap} other screen , which is fine . Um {gap} we agreed that the colour should be b uh yellow and black . Yellow in the back because it's m trendy , more trendy than black anyway . So {vocalsound} then we ca yeah . We agreed that we would imple

In [81]:
# Qwen3_Embedding

model = Qwen3_Embedding(512)

In [82]:
model.encode([test_text[:5000]], None, None, None)

Token indices sequence length is longer than the specified maximum sequence length for this model (1437 > 512). Running this sequence through the model will result in indexing errors


tensor([[-0.3841, -6.5106, -1.0276,  ..., -1.4713,  0.8210, -0.0231]])

In [83]:
# XMLRoBERTa

model = XMLRoBERTa(512)

In [84]:
model.encode([test_text[:5000]], None, None, None)

tensor([[ 0.0197, -0.0395,  0.0856,  ...,  0.0039,  0.0753, -0.0517]])